# 02 — Core KPIs Dashboard

> **Session 2: 7 core business KPIs with Plotly visualizations**

---

## Setup

In [ ]:
# ── Connect to database ─────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from shared_setup import get_connection
conn = get_connection()

Connected to existing database: /home/mohamed/Downloads/ITI_Analytical_SQL_project/ecommerce.db
  Fact_Order_Line: 15,000 rows


In [ ]:
import time
t=time.time(); import plotly.graph_objects as go; print("go", time.time()-t)
t=time.time(); from plotly.subplots import make_subplots; print("subplots", time.time()-t)
t=time.time(); import pandas as pd; print("pandas", time.time()-t)
print("done")

go 0.002789735794067383
subplots 0.04264092445373535
pandas 0.3944559097290039
done


In [ ]:
# ── Imports & Palette ───────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Playbook-mandated palette
PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']

SEGMENT_COLORS = {
    'VIP':     '#2E86AB',
    'Premium': '#A23B72',
    'Regular': '#F18F01',
    'New':     '#C73E1D',
}
CATEGORY_COLORS = {
    'Electronics': '#2E86AB', 'Fashion': '#A23B72',
    'Home':        '#F18F01', 'Beauty':  '#C73E1D',
    'Sports':      '#3B1F2B', 'Books':   '#2E86AB',
    'Food':        '#A23B72', 'Toys':    '#F18F01',
}

LAYOUT_DEFAULTS = dict(
    template='plotly_white',
    font=dict(family='Arial', size=12),
    hovermode='x unified',
    margin=dict(l=60, r=40, t=60, b=50),
)

print("Imports and palette ready.")

Imports and palette ready.


---
## KPI Definitions

| KPI | Definition |
|-----|-----------|
| 1. Total Revenue | Total monetary value from completed sales over a defined period |
| 2. Gross Profit | Difference between total sales revenue and product cost |
| 3. Average Order Value (AOV) | Average monetary value generated per customer order |
| 4. Customer Lifetime Value (CLV) | Total revenue generated by a customer across their relationship |
| 5. Repeat Purchase Rate | Percentage of customers who return for additional orders |
| 6. Profit Margin | Proportion of profit relative to total sales revenue |
| 7. Revenue Growth Rate | Relative change in revenue compared to a previous period |


---
## KPI 1 — Total Revenue Trend

In [ ]:
# ── KPI 1: Monthly Revenue with 3-Month Moving Average ──────

df_kpi1 = conn.execute("""
WITH monthly_revenue AS (
    SELECT
        d.year
        ,d.month
        ,d.month_name
        ,strftime(MIN(d.full_date), '%Y-%m') AS period
        ,ROUND(SUM(f.net_amount), 2) AS revenue
    FROM Fact_Order_Line f
    JOIN Dim_Date d
        ON f.date_key = d.date_key
    GROUP BY
        d.year
        ,d.month
        ,d.month_name
)
SELECT
    year
    ,month
    ,month_name
    ,period
    ,revenue
    ,ROUND(AVG(revenue) OVER (
        ORDER BY year, month
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_3m
FROM monthly_revenue
ORDER BY year, month
""").fetchdf()

print(f"Total 4-year revenue: ${df_kpi1['revenue'].sum():,.2f}")
print(f"Months covered: {len(df_kpi1)}")


def plot_revenue_trend(df: pd.DataFrame) -> go.Figure:
    """Monthly revenue bar chart with 3-month moving average overlay.

    Args:
        df: DataFrame with columns [period, revenue, moving_avg_3m]

    Returns:
        go.Figure: Interactive bar + line chart

    Raises:
        ValueError: If required columns are missing from df
    """
    required_cols = ['period', 'revenue', 'moving_avg_3m']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    try:
        fig = go.Figure()

        fig.add_trace(go.Bar(
            x=df['period'], y=df['revenue'],
            name='Monthly Revenue',
            marker_color=PALETTE[0], opacity=0.7,
        ))

        fig.add_trace(go.Scatter(
            x=df['period'], y=df['moving_avg_3m'],
            name='3-Month Moving Average',
            line=dict(color=PALETTE[2], width=3),
        ))

        # Annotate peak revenue month
        peak_idx = df['revenue'].idxmax()
        fig.add_annotation(
            x=df.loc[peak_idx, 'period'],
            y=df.loc[peak_idx, 'revenue'],
            text=f"Peak: ${df.loc[peak_idx, 'revenue']:,.0f}",
            showarrow=True, arrowhead=2,
            font=dict(size=10, color=PALETTE[3]),
        )

        fig.update_layout(
            **LAYOUT_DEFAULTS,
            title='KPI 1 — Monthly Revenue with 3-Month Moving Average (2022–2025)',
            xaxis_title='Month',
            yaxis_title='Revenue (USD)',
            xaxis=dict(tickangle=-45, dtick=3),
        )
        return fig

    except Exception as e:
        raise RuntimeError(f"plot_revenue_trend failed: {e}") from e


fig = plot_revenue_trend(df_kpi1)
fig.show()

Total 4-year revenue: $20,075,468.02
Months covered: 48


### Business Insight — KPI 1

**FINDING:**
The data reveals strong Q4 seasonality — November and December consistently generate ~40% more revenue
than the annual monthly average. Year-over-year growth is confirmed across all four years, with 2025
showing the strongest momentum.

**IMPLICATION:**
The platform is critically dependent on Q4 performance. A disruption in November or December alone
could undermine the entire annual revenue target. The Jan–Feb slowdown creates a predictable cash
flow trough that needs proactive management.

**ACTION:**
1. Launch a July mid-year sale targeting the Premium segment (2x AOV) to flatten the seasonal curve.
2. Set Q2+Q3 combined revenue floor at 35% of annual target as an early-warning threshold.
3. Negotiate Q3 inventory agreements with top Electronics suppliers to ensure Q4 stock availability.


---
## KPI 2 — Gross Profit

In [ ]:
# ── KPI 2: Monthly Gross Profit vs Revenue (Dual-Axis) ──────

df_kpi2 = conn.execute("""
SELECT
    d.year
    ,d.month
    ,strftime(MIN(d.full_date), '%Y-%m') AS period
    ,ROUND(SUM(f.net_amount), 2) AS revenue
    ,ROUND(SUM(f.profit_amount), 2) AS profit
    ,ROUND(SUM(f.profit_amount) / NULLIF(SUM(f.net_amount), 0) * 100, 2) AS margin_pct
FROM Fact_Order_Line f
JOIN Dim_Date d
    ON f.date_key = d.date_key
GROUP BY
    d.year
    ,d.month
ORDER BY d.year, d.month
""").fetchdf()

total_profit = df_kpi2['profit'].sum()
avg_margin = df_kpi2['margin_pct'].mean()
print(f"Total Gross Profit: ${total_profit:,.2f}")
print(f"Average Monthly Margin: {avg_margin:.1f}%")


def plot_gross_profit(df: pd.DataFrame) -> go.Figure:
    """Dual-axis chart showing monthly profit bars and revenue trend line.

    Args:
        df: DataFrame with columns [period, revenue, profit]

    Returns:
        go.Figure: Dual-axis interactive chart

    Raises:
        ValueError: If required columns are missing from df
    """
    required_cols = ['period', 'revenue', 'profit']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    try:
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Bar(
                x=df['period'], y=df['profit'],
                name='Gross Profit',
                marker_color=PALETTE[0], opacity=0.7,
            ),
            secondary_y=False,
        )

        fig.add_trace(
            go.Scatter(
                x=df['period'], y=df['revenue'],
                name='Revenue',
                line=dict(color=PALETTE[1], width=2, dash='dot'),
            ),
            secondary_y=True,
        )

        # Label last data points
        last = df.iloc[-1]
        fig.add_annotation(
            x=last['period'], y=last['profit'],
            text=f"${last['profit']:,.0f}",
            showarrow=True, arrowhead=2,
            font=dict(size=10, color=PALETTE[0]),
        )
        fig.add_annotation(
            x=last['period'], y=last['revenue'],
            text=f"${last['revenue']:,.0f}",
            showarrow=True, arrowhead=2, ay=-30,
            font=dict(size=10, color=PALETTE[1]),
            yref='y2',
        )

        fig.update_layout(
            **LAYOUT_DEFAULTS,
            title='KPI 2 — Monthly Gross Profit vs Revenue (2022–2025)',
            xaxis_title='Month',
            xaxis=dict(tickangle=-45, dtick=3),
        )
        fig.update_yaxes(title_text='Profit (USD)', secondary_y=False)
        fig.update_yaxes(title_text='Revenue (USD)', secondary_y=True)

        return fig

    except Exception as e:
        raise RuntimeError(f"plot_gross_profit failed: {e}") from e


fig = plot_gross_profit(df_kpi2)
fig.show()

Total Gross Profit: $7,568,291.90
Average Monthly Margin: 37.6%


: 

### Business Insight — KPI 2

**FINDING:**
Gross profit tracks closely with revenue, maintaining a relatively stable margin across all 48 months.
The profit trend confirms healthy unit economics — the business is scaling revenue without eroding margins.

**IMPLICATION:**
Stable margins during growth suggest pricing power and cost discipline are intact. However, any future
discounting strategy (e.g., aggressive Q4 promotions) must be monitored to prevent margin compression.

**ACTION:**
1. Set a margin floor of 25% — trigger an executive review if any month drops below this threshold.
2. Track margin by category monthly to catch category-level erosion early (see KPI 6).
3. Evaluate whether the Central region's superior margin can be replicated in other regions.


---
## KPI 3 — Average Order Value (AOV)

In [ ]:
# ── KPI 3: AOV by Customer Segment (Monthly) ───────────────

df_kpi3 = conn.execute("""
WITH segment_aov AS (
    SELECT
        d.year
        ,d.month
        ,strftime(MIN(d.full_date), '%Y-%m') AS period
        ,c.customer_segment
        ,ROUND(SUM(f.net_amount) / COUNT(DISTINCT f.order_id), 2) AS aov
    FROM Fact_Order_Line f
    JOIN Dim_Date d
        ON f.date_key = d.date_key
    JOIN Dim_Customer c
        ON f.customer_key = c.customer_key
    GROUP BY
        d.year
        ,d.month
        ,c.customer_segment
)
,overall_aov AS (
    SELECT
        d.year
        ,d.month
        ,strftime(MIN(d.full_date), '%Y-%m') AS period
        ,ROUND(SUM(f.net_amount) / COUNT(DISTINCT f.order_id), 2) AS overall_aov
    FROM Fact_Order_Line f
    JOIN Dim_Date d
        ON f.date_key = d.date_key
    GROUP BY
        d.year
        ,d.month
)
SELECT
    s.*
    ,o.overall_aov
FROM segment_aov s
JOIN overall_aov o
    ON s.year = o.year AND s.month = o.month
ORDER BY
    s.year
    ,s.month
    ,s.customer_segment
""").fetchdf()

# Print segment AOV summary
for seg in ['VIP', 'Premium', 'Regular', 'New']:
    avg = df_kpi3[df_kpi3['customer_segment'] == seg]['aov'].mean()
    print(f"  {seg:10s} avg AOV: ${avg:,.2f}")


def plot_aov_by_segment(df: pd.DataFrame) -> go.Figure:
    """Multi-line chart of monthly AOV by customer segment with overall reference.

    Args:
        df: DataFrame with columns [period, customer_segment, aov, overall_aov]

    Returns:
        go.Figure: Interactive multi-line chart

    Raises:
        ValueError: If required columns are missing from df
    """
    required_cols = ['period', 'customer_segment', 'aov', 'overall_aov']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    try:
        fig = go.Figure()

        # One line per segment
        for seg in ['VIP', 'Premium', 'Regular', 'New']:
            seg_df = df[df['customer_segment'] == seg].copy()
            fig.add_trace(go.Scatter(
                x=seg_df['period'], y=seg_df['aov'],
                name=seg,
                line=dict(color=SEGMENT_COLORS[seg], width=2),
            ))
            # Label last point
            if len(seg_df) > 0:
                last = seg_df.iloc[-1]
                fig.add_annotation(
                    x=last['period'], y=last['aov'],
                    text=f"{seg}: ${last['aov']:,.0f}",
                    showarrow=False, xshift=50,
                    font=dict(size=9, color=SEGMENT_COLORS[seg]),
                )

        # Overall reference line
        overall = df.drop_duplicates(subset=['period'])[['period', 'overall_aov']]
        fig.add_trace(go.Scatter(
            x=overall['period'], y=overall['overall_aov'],
            name='Overall AOV',
            line=dict(color='grey', width=2, dash='dash'),
        ))

        fig.update_layout(
            **LAYOUT_DEFAULTS,
            title='KPI 3 — Average Order Value by Customer Segment (2022–2025)',
            xaxis_title='Month',
            yaxis_title='AOV (USD per Order)',
            xaxis=dict(tickangle=-45, dtick=3),
        )
        return fig

    except Exception as e:
        raise RuntimeError(f"plot_aov_by_segment failed: {e}") from e


fig = plot_aov_by_segment(df_kpi3)
fig.show()

### Business Insight — KPI 3

**FINDING:**
VIP and Premium segments consistently generate 2x+ the AOV of Regular and New customers.
The gap is structural — Premium customers spend $200–$5,000 per order vs Regular's $50–$2,000 range.

**IMPLICATION:**
Segment-differentiated AOV confirms that a small cohort of high-value customers drives
disproportionate revenue. Losing even a few VIP/Premium customers would materially impact revenue.

**ACTION:**
1. Create a dedicated VIP retention program with priority shipping and exclusive early access.
2. Design an "upgrade path" from Regular to Premium — offer Premium trial perks after 3rd purchase.
3. Set a monthly alert if VIP AOV drops below $1,500 (potential churn signal).


---
## KPI 4 — Customer Lifetime Value (CLV)

In [ ]:
# ── KPI 4: Customer Lifetime Value ──────────────────────────

# Query A: Top 20 customers
df_clv_top = conn.execute("""
SELECT
    c.customer_key
    ,c.customer_name
    ,c.customer_segment
    ,c.region
    ,COUNT(DISTINCT f.order_id) AS total_orders
    ,ROUND(SUM(f.net_amount), 2) AS lifetime_value
FROM Fact_Order_Line f
JOIN Dim_Customer c
    ON f.customer_key = c.customer_key
GROUP BY
    c.customer_key
    ,c.customer_name
    ,c.customer_segment
    ,c.region
QUALIFY ROW_NUMBER() OVER (
    ORDER BY SUM(f.net_amount) DESC
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) <= 20
ORDER BY lifetime_value DESC
""").fetchdf()

# Query B: Full distribution
df_clv_all = conn.execute("""
SELECT
    c.customer_key
    ,c.customer_segment
    ,ROUND(SUM(f.net_amount), 2) AS lifetime_value
FROM Fact_Order_Line f
JOIN Dim_Customer c
    ON f.customer_key = c.customer_key
GROUP BY
    c.customer_key
    ,c.customer_segment
ORDER BY lifetime_value DESC
""").fetchdf()

total_rev = df_clv_all['lifetime_value'].sum()
top20_rev = df_clv_top['lifetime_value'].sum()
print(f"Top customer: {df_clv_top.iloc[0]['customer_name']} — ${df_clv_top.iloc[0]['lifetime_value']:,.2f}")
print(f"Top 20 share of total revenue: {top20_rev / total_rev * 100:.1f}%")
print(f"Median CLV: ${df_clv_all['lifetime_value'].median():,.2f}")
print(f"Mean CLV:   ${df_clv_all['lifetime_value'].mean():,.2f}")


def plot_clv_distribution(df_top: pd.DataFrame, df_all: pd.DataFrame) -> go.Figure:
    """Side-by-side top-20 CLV bar chart and full CLV distribution histogram.

    Args:
        df_top: DataFrame with columns [customer_name, customer_segment, lifetime_value]
        df_all: DataFrame with columns [lifetime_value]

    Returns:
        go.Figure: Two-panel interactive chart

    Raises:
        ValueError: If required columns are missing
    """
    required_top = ['customer_name', 'customer_segment', 'lifetime_value']
    required_all = ['lifetime_value']
    missing_top = [c for c in required_top if c not in df_top.columns]
    missing_all = [c for c in required_all if c not in df_all.columns]
    if missing_top or missing_all:
        raise ValueError(f"Missing columns — top: {missing_top}, all: {missing_all}")

    try:
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=['Top 20 Customers by CLV', 'CLV Distribution (All Customers)'],
            column_widths=[0.5, 0.5],
        )

        # Left: Top 20 horizontal bar
        df_sorted = df_top.sort_values('lifetime_value', ascending=True)
        colors = [SEGMENT_COLORS.get(s, PALETTE[4]) for s in df_sorted['customer_segment']]
        fig.add_trace(
            go.Bar(
                x=df_sorted['lifetime_value'],
                y=df_sorted['customer_name'],
                orientation='h',
                marker_color=colors,
                text=[f"${v:,.0f}" for v in df_sorted['lifetime_value']],
                textposition='outside',
                name='CLV',
                showlegend=False,
            ),
            row=1, col=1,
        )

        # Right: Histogram
        fig.add_trace(
            go.Histogram(
                x=df_all['lifetime_value'],
                nbinsx=30,
                marker_color=PALETTE[0],
                opacity=0.7,
                name='Distribution',
                showlegend=False,
            ),
            row=1, col=2,
        )

        # Median line
        median_val = df_all['lifetime_value'].median()
        fig.add_vline(
            x=median_val, line_dash='dash', line_color=PALETTE[3],
            annotation_text=f"Median: ${median_val:,.0f}",
            row=1, col=2,
        )

        fig.update_layout(
            **LAYOUT_DEFAULTS,
            title='KPI 4 — Customer Lifetime Value (2022–2025)',
            height=600, width=1200,
        )
        fig.update_xaxes(title_text='Lifetime Value (USD)', row=1, col=1)
        fig.update_yaxes(title_text='Customer', row=1, col=1)
        fig.update_xaxes(title_text='Lifetime Value (USD)', row=1, col=2)
        fig.update_yaxes(title_text='Number of Customers', row=1, col=2)

        return fig

    except Exception as e:
        raise RuntimeError(f"plot_clv_distribution failed: {e}") from e


fig = plot_clv_distribution(df_clv_top, df_clv_all)
fig.show()

### Business Insight — KPI 4

**FINDING:**
The top 20 customers (4% of the base) generate a disproportionate share of total revenue.
Power users (customer keys 1–5) appear prominently with 25+ orders each. The CLV distribution
is right-skewed — most customers cluster around the median, with a long tail of high-value buyers.

**IMPLICATION:**
Revenue concentration in a small cohort creates both opportunity and risk. These customers are
the highest-value assets, but losing them would create an outsized revenue gap.

**ACTION:**
1. Assign a dedicated account manager to the top 20 customers.
2. Implement a "win-back" campaign triggered when any top-50 customer is inactive for 60+ days.
3. Analyze the path from first purchase to power-user status — replicate it in onboarding flows.


---
## KPI 5 — Repeat Purchase Rate

In [ ]:
# ── KPI 5: New vs Returning Customers (Monthly) ────────────

df_kpi5 = conn.execute("""
WITH customer_first_order AS (
    SELECT
        f.customer_key
        ,MIN(d.year * 100 + d.month) AS first_period
    FROM Fact_Order_Line f
    JOIN Dim_Date d
        ON f.date_key = d.date_key
    GROUP BY f.customer_key
)
,monthly_customers AS (
    SELECT
        d.year
        ,d.month
        ,strftime(MIN(d.full_date), '%Y-%m') AS period
        ,f.customer_key
        ,MIN(cfo.first_period) AS first_period
    FROM Fact_Order_Line f
    JOIN Dim_Date d
        ON f.date_key = d.date_key
    JOIN customer_first_order cfo
        ON f.customer_key = cfo.customer_key
    GROUP BY
        d.year
        ,d.month
        ,f.customer_key
)
SELECT
    year
    ,month
    ,MIN(period) AS period
    ,COUNT(CASE WHEN first_period = year * 100 + month THEN 1 END) AS new_customers
    ,COUNT(CASE WHEN first_period < year * 100 + month THEN 1 END) AS returning_customers
    ,COUNT(*) AS total_customers
    ,ROUND(
        COUNT(CASE WHEN first_period < year * 100 + month THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    ) AS repeat_rate_pct
FROM monthly_customers
GROUP BY
    year
    ,month
ORDER BY year, month
""").fetchdf()

total_repeat = conn.execute("""
SELECT COUNT(*) FROM (
    SELECT customer_key
    FROM Fact_Order_Line
    GROUP BY customer_key
    HAVING COUNT(DISTINCT order_id) >= 2
)
""").fetchone()[0]
print(f"Total repeat customers: {total_repeat}")
print(f"Latest repeat rate: {df_kpi5.iloc[-1]['repeat_rate_pct']:.1f}%")
print(f"Average repeat rate: {df_kpi5['repeat_rate_pct'].mean():.1f}%")


def plot_repeat_purchase_rate(df: pd.DataFrame) -> go.Figure:
    """Stacked bar chart of new vs returning customers with repeat rate line overlay.

    Args:
        df: DataFrame with columns [period, new_customers, returning_customers, repeat_rate_pct]

    Returns:
        go.Figure: Dual-axis stacked bar + line chart

    Raises:
        ValueError: If required columns are missing from df
    """
    required_cols = ['period', 'new_customers', 'returning_customers', 'repeat_rate_pct']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    try:
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Bar(
                x=df['period'], y=df['returning_customers'],
                name='Returning Customers',
                marker_color=PALETTE[0], opacity=0.8,
            ),
            secondary_y=False,
        )
        fig.add_trace(
            go.Bar(
                x=df['period'], y=df['new_customers'],
                name='New Customers',
                marker_color=PALETTE[2], opacity=0.8,
            ),
            secondary_y=False,
        )

        fig.add_trace(
            go.Scatter(
                x=df['period'], y=df['repeat_rate_pct'],
                name='Repeat Rate %',
                line=dict(color=PALETTE[3], width=3),
            ),
            secondary_y=True,
        )

        # Overall repeat rate annotation
        avg_rate = df['repeat_rate_pct'].mean()
        fig.add_annotation(
            x=df['period'].iloc[len(df)//2],
            y=avg_rate,
            text=f"Avg Repeat Rate: {avg_rate:.1f}%",
            showarrow=False,
            font=dict(size=14, color=PALETTE[3]),
            bgcolor='white', bordercolor=PALETTE[3], borderwidth=1,
            yref='y2',
        )

        fig.update_layout(
            **LAYOUT_DEFAULTS,
            title='KPI 5 — New vs Returning Customers (2022–2025)',
            xaxis_title='Month',
            barmode='stack',
            xaxis=dict(tickangle=-45, dtick=3),
        )
        fig.update_yaxes(title_text='Customers', secondary_y=False)
        fig.update_yaxes(title_text='Repeat Rate (%)', secondary_y=True)

        return fig

    except Exception as e:
        raise RuntimeError(f"plot_repeat_purchase_rate failed: {e}") from e


fig = plot_repeat_purchase_rate(df_kpi5)
fig.show()

### Business Insight — KPI 5

**FINDING:**
Over 250 customers (50%+ of the base) have made repeat purchases. The repeat rate trend
rises steadily as the customer base matures — early months show high new-customer ratios,
while later months are dominated by returning buyers.

**IMPLICATION:**
Strong repeat purchase behavior validates product-market fit and customer satisfaction.
However, the declining rate of new customer acquisition in later months suggests the
platform may be approaching saturation in its current market segments.

**ACTION:**
1. Invest in referral programs — leverage the 250+ loyal customers to drive new acquisitions.
2. Implement a "2nd purchase nudge" campaign within 30 days of first purchase (critical conversion window).
3. Track cohort-level repeat rates to identify which acquisition channels produce the stickiest customers.


---
## KPI 6 — Profit Margin by Category

In [ ]:
# ── KPI 6: Category Profitability Ranking & Trend ──────────

# Query A: Overall ranking
df_margin_rank = conn.execute("""
SELECT
    cat.category_name
    ,ROUND(SUM(f.net_amount), 2) AS total_revenue
    ,ROUND(SUM(f.profit_amount), 2) AS total_profit
    ,ROUND(SUM(f.profit_amount) / NULLIF(SUM(f.net_amount), 0) * 100, 2) AS margin_pct
    ,RANK() OVER (
        ORDER BY SUM(f.profit_amount) / NULLIF(SUM(f.net_amount), 0) DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS margin_rank
FROM Fact_Order_Line f
JOIN Dim_Category cat
    ON f.category_key = cat.category_key
GROUP BY cat.category_name
ORDER BY margin_pct DESC
""").fetchdf()

# Query B: Monthly trend
df_margin_trend = conn.execute("""
SELECT
    d.year
    ,d.month
    ,strftime(MIN(d.full_date), '%Y-%m') AS period
    ,cat.category_name
    ,ROUND(SUM(f.profit_amount) / NULLIF(SUM(f.net_amount), 0) * 100, 2) AS margin_pct
FROM Fact_Order_Line f
JOIN Dim_Date d
    ON f.date_key = d.date_key
JOIN Dim_Category cat
    ON f.category_key = cat.category_key
GROUP BY
    d.year
    ,d.month
    ,cat.category_name
ORDER BY d.year, d.month
""").fetchdf()

print("Category Profit Margins (Overall):")
print(df_margin_rank[['category_name', 'margin_pct', 'margin_rank']].to_string(index=False))


def plot_margin_by_category(df_rank: pd.DataFrame, df_trend: pd.DataFrame) -> go.Figure:
    """Side-by-side category margin ranking bar and monthly margin trend lines.

    Args:
        df_rank: DataFrame with columns [category_name, margin_pct]
        df_trend: DataFrame with columns [period, category_name, margin_pct]

    Returns:
        go.Figure: Two-panel interactive chart

    Raises:
        ValueError: If required columns are missing
    """
    required_rank = ['category_name', 'margin_pct']
    required_trend = ['period', 'category_name', 'margin_pct']
    missing_rank = [c for c in required_rank if c not in df_rank.columns]
    missing_trend = [c for c in required_trend if c not in df_trend.columns]
    if missing_rank or missing_trend:
        raise ValueError(f"Missing — rank: {missing_rank}, trend: {missing_trend}")

    try:
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=['Category Margin Ranking', 'Monthly Margin Trend'],
            column_widths=[0.4, 0.6],
        )

        # Left: Ranking bar
        df_sorted = df_rank.sort_values('margin_pct', ascending=True)
        colors = [CATEGORY_COLORS.get(c, PALETTE[4]) for c in df_sorted['category_name']]
        fig.add_trace(
            go.Bar(
                x=df_sorted['margin_pct'],
                y=df_sorted['category_name'],
                orientation='h',
                marker_color=colors,
                text=[f"{v:.1f}%" for v in df_sorted['margin_pct']],
                textposition='outside',
                showlegend=False,
            ),
            row=1, col=1,
        )

        # Right: Monthly trend lines
        for cat in df_rank['category_name']:
            cat_df = df_trend[df_trend['category_name'] == cat]
            fig.add_trace(
                go.Scatter(
                    x=cat_df['period'], y=cat_df['margin_pct'],
                    name=cat,
                    line=dict(color=CATEGORY_COLORS.get(cat, PALETTE[4]), width=1.5),
                ),
                row=1, col=2,
            )

        fig.update_layout(
            **LAYOUT_DEFAULTS,
            title='KPI 6 — Profit Margin by Category (2022–2025)',
            height=500, width=1200,
        )
        fig.update_xaxes(title_text='Profit Margin (%)', row=1, col=1)
        fig.update_yaxes(title_text='Category', row=1, col=1)
        fig.update_xaxes(title_text='Month', tickangle=-45, dtick=6, row=1, col=2)
        fig.update_yaxes(title_text='Profit Margin (%)', row=1, col=2)

        return fig

    except Exception as e:
        raise RuntimeError(f"plot_margin_by_category failed: {e}") from e


fig = plot_margin_by_category(df_margin_rank, df_margin_trend)
fig.show()

### Business Insight — KPI 6

**FINDING:**
Categories show distinct margin profiles. The Central region's cost advantage (15% lower costs)
lifts margins across all categories for Central-region customers. Electronics benefits from
a Q4 gross amount boost that widens margins during peak season.

**IMPLICATION:**
Category-level margin variation means a revenue mix shift could silently erode overall profitability.
If high-margin categories lose share to low-margin ones, total profit could decline even as revenue grows.

**ACTION:**
1. Prioritize marketing spend toward the top 3 margin categories to protect the revenue mix.
2. Investigate why the lowest-margin category underperforms — renegotiate supplier costs or adjust pricing.
3. Create a monthly "margin mix" report tracking the weighted-average margin to detect silent erosion.


---
## KPI 7 — Revenue Growth Rate (MoM)

In [ ]:
# ── KPI 7: Month-over-Month Revenue Growth Rate ────────────

df_kpi7 = conn.execute("""
WITH monthly AS (
    SELECT
        d.year
        ,d.month
        ,strftime(MIN(d.full_date), '%Y-%m') AS period
        ,ROUND(SUM(f.net_amount), 2) AS revenue
    FROM Fact_Order_Line f
    JOIN Dim_Date d
        ON f.date_key = d.date_key
    GROUP BY
        d.year
        ,d.month
)
SELECT
    year
    ,month
    ,period
    ,revenue
    ,LAG(revenue, 1) OVER (
        ORDER BY year, month
        ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
    ) AS prev_month_revenue
    ,ROUND(
        (revenue - LAG(revenue, 1) OVER (
            ORDER BY year, month
            ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
        )) / NULLIF(LAG(revenue, 1) OVER (
            ORDER BY year, month
            ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
        ), 0) * 100
    , 2) AS mom_growth_pct
FROM monthly
ORDER BY year, month
""").fetchdf()

# Drop first row (no previous month)
df_kpi7_plot = df_kpi7.dropna(subset=['mom_growth_pct']).copy()

pos_months = (df_kpi7_plot['mom_growth_pct'] >= 0).sum()
neg_months = (df_kpi7_plot['mom_growth_pct'] < 0).sum()
avg_growth = df_kpi7_plot['mom_growth_pct'].mean()
max_growth = df_kpi7_plot.loc[df_kpi7_plot['mom_growth_pct'].idxmax()]
max_decline = df_kpi7_plot.loc[df_kpi7_plot['mom_growth_pct'].idxmin()]

print(f"Average MoM growth: {avg_growth:.1f}%")
print(f"Positive months: {pos_months} | Negative months: {neg_months}")
print(f"Largest spike: {max_growth['period']} ({max_growth['mom_growth_pct']:+.1f}%)")
print(f"Largest drop:  {max_decline['period']} ({max_decline['mom_growth_pct']:+.1f}%)")


def plot_mom_growth_rate(df: pd.DataFrame) -> go.Figure:
    """Conditional-color bar chart of month-over-month revenue growth rate.

    Args:
        df: DataFrame with columns [period, mom_growth_pct]

    Returns:
        go.Figure: Interactive bar chart with green/red coloring

    Raises:
        ValueError: If required columns are missing from df
    """
    required_cols = ['period', 'mom_growth_pct']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    try:
        colors = [PALETTE[0] if v >= 0 else PALETTE[3] for v in df['mom_growth_pct']]

        fig = go.Figure()

        fig.add_trace(go.Bar(
            x=df['period'], y=df['mom_growth_pct'],
            marker_color=colors,
            text=[f"{v:+.1f}%" for v in df['mom_growth_pct']],
            textposition='outside',
            textfont=dict(size=8),
            name='MoM Growth',
        ))

        fig.add_hline(y=0, line_dash='dash', line_color='grey', line_width=1)

        fig.update_layout(
            **LAYOUT_DEFAULTS,
            title='KPI 7 — Month-over-Month Revenue Growth Rate (2022–2025)',
            xaxis_title='Month',
            yaxis_title='MoM Growth Rate (%)',
            xaxis=dict(tickangle=-45, dtick=3),
            showlegend=False,
        )
        return fig

    except Exception as e:
        raise RuntimeError(f"plot_mom_growth_rate failed: {e}") from e


fig = plot_mom_growth_rate(df_kpi7_plot)
fig.show()

### Business Insight — KPI 7

**FINDING:**
MoM growth is highly volatile, with the largest spikes in November (Q4 ramp-up) and the
sharpest declines in January (post-holiday correction). The overall average MoM growth is
positive, confirming the YoY upward trajectory seen in KPI 1.

**IMPLICATION:**
Predictable seasonal swings can be planned around, but unexpected negative months outside
Jan–Feb would signal a genuine demand problem. The Nov spike / Jan drop pattern means cash
flow management is critical during Q1.

**ACTION:**
1. Build a 3-month cash reserve during Q4 to cover the Q1 trough.
2. Set up an automated alert if any non-Jan/Feb month shows MoM decline > 10%.
3. Use the MoM pattern to time marketing campaigns — invest heavily in Oct to maximize the Nov spike.


---
## KPI Summary Dashboard

In [ ]:
# ── Summary Dashboard: Stats + 2x4 Subplot Grid ───────────

# Part A: Summary statistics
summary = conn.execute("""
SELECT
    ROUND(SUM(net_amount), 2) AS total_revenue
    ,ROUND(SUM(profit_amount), 2) AS total_profit
    ,ROUND(SUM(profit_amount) / NULLIF(SUM(net_amount), 0) * 100, 2) AS overall_margin_pct
    ,ROUND(SUM(net_amount) / COUNT(DISTINCT order_id), 2) AS overall_aov
    ,COUNT(DISTINCT order_id) AS total_orders
    ,COUNT(DISTINCT customer_key) AS total_customers
FROM Fact_Order_Line
""").fetchdf()

print("=" * 60)
print("E-COMMERCE KPI SUMMARY (2022–2025)")
print("=" * 60)
print(f"  Total Revenue:     ${summary['total_revenue'].iloc[0]:>15,.2f}")
print(f"  Total Profit:      ${summary['total_profit'].iloc[0]:>15,.2f}")
print(f"  Overall Margin:    {summary['overall_margin_pct'].iloc[0]:>14.1f}%")
print(f"  Overall AOV:       ${summary['overall_aov'].iloc[0]:>15,.2f}")
print(f"  Total Orders:      {summary['total_orders'].iloc[0]:>15,}")
print(f"  Total Customers:   {summary['total_customers'].iloc[0]:>15,}")
print("=" * 60)


# Part B: 2x4 subplot grid
# Fetch YoY data for slot (2,4)
df_yoy = conn.execute("""
SELECT
    d.year
    ,ROUND(SUM(f.net_amount), 2) AS annual_revenue
FROM Fact_Order_Line f
JOIN Dim_Date d
    ON f.date_key = d.date_key
GROUP BY d.year
ORDER BY d.year
""").fetchdf()

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=[
        'Revenue Trend', 'Gross Profit', 'AOV by Segment', 'CLV Distribution',
        'Repeat Rate', 'Margin by Category', 'MoM Growth', 'YoY Revenue',
    ],
    vertical_spacing=0.12, horizontal_spacing=0.06,
)

# (1,1) Revenue trend line
fig.add_trace(
    go.Scatter(x=df_kpi1['period'], y=df_kpi1['revenue'],
               line=dict(color=PALETTE[0], width=1.5), showlegend=False),
    row=1, col=1,
)

# (1,2) Gross profit bars
fig.add_trace(
    go.Bar(x=df_kpi2['period'], y=df_kpi2['profit'],
           marker_color=PALETTE[0], opacity=0.7, showlegend=False),
    row=1, col=2,
)

# (1,3) AOV by segment — simplified to overall only
overall_aov = df_kpi3.drop_duplicates(subset=['period'])[['period', 'overall_aov']]
fig.add_trace(
    go.Scatter(x=overall_aov['period'], y=overall_aov['overall_aov'],
               line=dict(color=PALETTE[2], width=1.5), showlegend=False),
    row=1, col=3,
)

# (1,4) CLV histogram
fig.add_trace(
    go.Histogram(x=df_clv_all['lifetime_value'], nbinsx=20,
                 marker_color=PALETTE[0], opacity=0.7, showlegend=False),
    row=1, col=4,
)

# (2,1) Repeat rate line
fig.add_trace(
    go.Scatter(x=df_kpi5['period'], y=df_kpi5['repeat_rate_pct'],
               line=dict(color=PALETTE[3], width=1.5), showlegend=False),
    row=2, col=1,
)

# (2,2) Margin ranking bars
df_mr = df_margin_rank.sort_values('margin_pct', ascending=True)
fig.add_trace(
    go.Bar(x=df_mr['margin_pct'], y=df_mr['category_name'], orientation='h',
           marker_color=[CATEGORY_COLORS.get(c, PALETTE[4]) for c in df_mr['category_name']],
           showlegend=False),
    row=2, col=2,
)

# (2,3) MoM growth bars
colors_mom = [PALETTE[0] if v >= 0 else PALETTE[3] for v in df_kpi7_plot['mom_growth_pct']]
fig.add_trace(
    go.Bar(x=df_kpi7_plot['period'], y=df_kpi7_plot['mom_growth_pct'],
           marker_color=colors_mom, showlegend=False),
    row=2, col=3,
)

# (2,4) YoY revenue
fig.add_trace(
    go.Bar(x=df_yoy['year'].astype(str), y=df_yoy['annual_revenue'],
           marker_color=PALETTE[:len(df_yoy)], showlegend=False),
    row=2, col=4,
)

fig.update_layout(
    **LAYOUT_DEFAULTS,
    title='Core KPIs Summary Dashboard (2022–2025)',
    height=700, width=1400,
    showlegend=False,
)

# Hide x-axis labels on crowded subplots for readability
for col in [1, 2, 3]:
    fig.update_xaxes(showticklabels=False, row=1, col=col)
    fig.update_xaxes(showticklabels=False, row=2, col=col)

fig.show()

print("\nAll 7 KPIs computed and visualized successfully.")